# 52 — ATS Simulation Mode
**Goal:** Simulate how a real ATS processes and scores a resume.

So far the pieces were built in isolation; this chapter wires them into one **pipeline**. `ATSSimulator` runs the four stages a real ATS performs — parse the document into sections, extract skills from a canonical vocabulary, score with the Ch. 51 explainable scorer, and print a report — as a single visible, reproducible pass over one resume + JD pair.

**Why it matters for resumes / ATS:** a simulation is the cheapest way to test the whole system before it touches real candidates: every stage's intermediate state (detected sections, extracted skills, per-dimension reasons) is visible, which is exactly what you need for demos and for debugging a rule change. It is also the first honest end-to-end view of what a recruiter's ATS would actually see.

## 1. Full ATS Pipeline

`ATSSimulator.run(resume_text, jd_text)` is a scripted four-step pipeline that prints its own progress and returns a structured result dict.

**What the code does:**
- **Parse** — scans for six section keywords with word-boundary regexes and prints `Sections detected: [...]`.
- **Extract** — matches resume and JD text against a 10-skill `skills_db` by case-insensitive substring, so "Python" is picked up regardless of formatting.
- **Score** — delegates to `ExplainableScorer.score_with_explanations()` (Ch. 51), with `must_have_terms` set to the first three JD skills.
- **Report** — prints `FINAL ATS SCORE` plus one line per dimension with its reason, and returns `{score, dimensions, skills}` for programmatic use.

**Expected (verified by running):** on the sample resume/JD the run reports `FINAL ATS SCORE: 64.0/100` with `skill_match 100.0` (4/4), `experience 100.0`, `format 40.0`, `boolean 100.0` (3/3). Two surprises worth learning from: `Sections detected: []` despite the resume containing `SUMMARY`/`EXPERIENCE`/`SKILLS`/`EDUCATION` — the same `r"\\b"` word-boundary trap from Ch. 50, which also explains `format 40.0` (four sections counted missing, 4 × 15 deducted) — and the 64.0 total is out of a 70-point budget because `education`, `bullet_quality`, and `duration` are never computed. **Prerequisite:** this cell assumes `ATSScorer`, `ExplainableScorer`, and `re` are already in the kernel from Ch. 50–51; run those first or the class definition raises `NameError`.

In [ ]:
class ATSSimulator:
    def __init__(self):
        self.scorer = ExplainableScorer()
        self.skills_db = ["Python", "TensorFlow", "PyTorch", "NLP", "SQL", "AWS", "Docker", "Kubernetes", "Spark", "Java"]
    
    def run(self, resume_text, jd_text):
        print("=" * 60)
        print("ATS SIMULATION".center(60))
        print("=" * 60)
        
        # Step 1: Parse resume
        print("\n1. PARSING RESUME...")
        sections_found = []
        for sec in ["summary", "experience", "education", "skills", "projects", "certifications"]:
            if re.search(r"\\b" + sec + r"\\b", resume_text, re.IGNORECASE):
                sections_found.append(sec)
        print(f"   Sections detected: {sections_found}")
        
        # Step 2: Extract skills
        print("\n2. EXTRACTING SKILLS...")
        resume_skills = [s for s in self.skills_db if s.lower() in resume_text.lower()]
        jd_skills = [s for s in self.skills_db if s.lower() in jd_text.lower()]
        print(f"   Resume skills: {resume_skills}")
        print(f"   JD requires: {jd_skills}")
        
        # Step 3: Score
        print("\n3. SCORING...")
        total, dims = self.scorer.score_with_explanations(
            resume_text, jd_text, resume_skills, jd_skills,
            must_have_terms=jd_skills[:3]
        )
        
        # Step 4: Results
        print("\n4. RESULTS")
        print(f"   FINAL ATS SCORE: {total}/100")
        for dim, exp in dims.items():
            print(f"   {dim:15s}: {exp['score']:5.1f}/100 - {exp['reason']}")
        
        return {"score": total, "dimensions": dims, "skills": {"resume": resume_skills, "jd": jd_skills}}

sim = ATSSimulator()
resume = """SUMMARY
Senior data scientist with 5+ years experience.

EXPERIENCE
Google — Senior Data Scientist, 2020-Present
- Developed ML pipelines in Python and TensorFlow
- Reduced inference latency by 40%

SKILLS
Python, NLP, TensorFlow, SQL, AWS

EDUCATION
M.S. Computer Science, Stanford University
"""
jd = """Senior Data Scientist
Requirements: Python, NLP, TensorFlow, SQL. 5+ years experience preferred.
"""
sim.run(resume, jd)

## Summary: ATS simulation makes the scoring pipeline visible end-to-end. Useful for debugging and demos.

**A pipeline you can watch is a pipeline you can fix — simulation turns scoring into a visible, repeatable run.**

With parse → extract → score → report in one method, every intermediate value (detected sections, extracted skills, dimension reasons) is inspectable, and the same run that scores 64.0 also surfaces the two live bugs in the rule set: the broken word-boundary regex and the missing dimension weights. That is Ch. 51's explainability applied at pipeline scale.

This chapter's extract stage feeds Ch. 53, where the extracted skill list is diffed against career-path requirements to find what is missing.